# 01 - Exploratory Data Analysis

Synthetic 12-domain enterprise policy corpus + held-out Q&A.
Goal of this notebook: confirm domain balance, paragraph-length distribution,
question difficulty, and a quick lexical-overlap sanity check between questions
and their ground-truth source paragraphs.

Run `python -m enterprise_rag.data` first to materialize the parquet files.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(context="notebook", style="whitegrid")
pd.set_option("display.max_colwidth", 120)

In [ ]:
corpus = pd.read_parquet("../data/processed/policy_corpus.parquet")
qa = pd.read_parquet("../data/processed/policy_qa.parquet")
print(f"corpus rows: {len(corpus):,}  | qa rows: {len(qa):,}")
corpus.head(3)

## 1. Schema & dtypes

In [ ]:
print(corpus.dtypes)
print("missing values per column:")
print(corpus.isna().sum())

## 2. Domain balance
We expect each of the 12 policy domains to contribute roughly 1/12 of the corpus.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
domain_counts = corpus["domain"].value_counts().sort_values()
domain_counts.plot(kind="barh", ax=ax, color="#3b82f6")
ax.set_title("Paragraph count by policy domain")
ax.set_xlabel("# paragraphs")
plt.tight_layout()
plt.show()

## 3. Paragraph length distribution

In [ ]:
corpus["n_words"] = corpus["text"].str.split().apply(len)
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(corpus["n_words"], bins=30, ax=ax, color="#10b981")
ax.set_title("Words per paragraph")
ax.set_xlabel("# words")
plt.tight_layout()
plt.show()
corpus["n_words"].describe()

## 4. Question length distribution

In [ ]:
qa["q_words"] = qa["question"].str.split().apply(len)
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(qa["q_words"], bins=20, ax=ax, color="#8b5cf6")
ax.set_title("Words per question")
ax.set_xlabel("# words")
plt.tight_layout()
plt.show()

## 5. Lexical overlap question vs. ground-truth source
A leakage / sanity check: the closer the overlap, the easier BM25 will have it.

In [ ]:
import re
TOKEN_RE = re.compile(r"[A-Za-z0-9]+")
def tokens(s):
    return set(t.lower() for t in TOKEN_RE.findall(str(s)))

key = corpus.set_index(["doc_id", "paragraph_id"])
def first_source_overlap(row):
    pieces = str(row["source_doc_ids"]).split(";")
    if not pieces:
        return np.nan
    d, p = pieces[0].split(":")
    if (d, int(p)) not in key.index:
        return np.nan
    paragraph = key.loc[(d, int(p)), "text"]
    q = tokens(row["question"])
    s = tokens(paragraph)
    return len(q & s) / max(len(q), 1)

qa["overlap"] = qa.apply(first_source_overlap, axis=1)
qa["overlap"].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(qa["overlap"].dropna(), bins=20, ax=ax, color="#f59e0b")
ax.set_title("Lexical overlap: question vs. first source paragraph")
ax.set_xlabel("|q tokens & source tokens| / |q tokens|")
plt.tight_layout()
plt.show()

## 6. Per-domain question count

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
qa["domain"].value_counts().sort_values().plot(kind="barh", ax=ax, color="#ef4444")
ax.set_title("Q&A count by domain")
plt.tight_layout()
plt.show()

## 7. Takeaways
- The 12 domains are balanced; per-domain recall is a meaningful slice metric.
- Paragraphs are short-to-medium length, well within BM25 / TF-IDF comfort zone.
- Lexical overlap is non-trivial but bounded — pure BM25 gets a head start, dense retrieval should help when the question rephrases the source vocabulary.